# 🗂️ Notebook 2: Netflix — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/netflix
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Core entities

| Table | Key columns |
|---|---|
| `users` | id, email, region |
| `titles` | id, name, synopsis, genres, release_year |
| `video_assets` | id, title_id, language, encoded_renditions |
| `rendition` | video_asset_id, resolution, bitrate_kbps, cdn_url |
| `watch_history` | user_id, title_id, position_seconds, updated_at |
| `subscriptions` | user_id, tier, valid_until |

### Why split `titles` from `video_assets`?
A movie (title) may have many *assets*: dubs in other languages, director's cut, etc. Each asset
is then encoded into many *renditions* (240p / 480p / 720p / 1080p / 4K).


## Key APIs

```http
# Browse
GET  /catalog/home              → personalized rails
GET  /titles/{id}               → title details
GET  /search?q=...

# Playback handshake
POST /playback/start            { title_id }
     → { manifest_url, session_token, initial_position }

# The *player* then fetches manifest_url from the CDN,
# and the CDN streams HLS chunks directly.

# Position checkpoints (low-priority, batched)
POST /history/position          { title_id, position_seconds }

# Recommendations
GET  /recs/for-me               → ranked list of titles
```

Important: the server returns a **manifest URL** pointing to a CDN, not raw bytes.
Our origin servers never stream video.


In [ ]:
# A minimal pydantic model of the API surface — runs without a server.
from pydantic import BaseModel
from datetime import datetime

class Rendition(BaseModel):
    resolution: str
    bitrate_kbps: int
    cdn_url: str

class Title(BaseModel):
    id: int
    name: str
    genres: list[str]

class PlaybackSession(BaseModel):
    session_token: str
    manifest_url: str
    initial_position_sec: int

class PositionCheckpoint(BaseModel):
    title_id: int
    position_seconds: int
    ts: datetime

# Example serialization the API would return
sess = PlaybackSession(session_token="sess-abc",
                       manifest_url="https://cdn.example.com/m/42.m3u8",
                       initial_position_sec=120)
print(sess.model_dump_json(indent=2))


## An HLS manifest (what the CDN actually returns)

```
#EXTM3U
#EXT-X-VERSION:3
#EXT-X-STREAM-INF:BANDWIDTH=800000,RESOLUTION=640x360
360p.m3u8
#EXT-X-STREAM-INF:BANDWIDTH=2800000,RESOLUTION=1280x720
720p.m3u8
#EXT-X-STREAM-INF:BANDWIDTH=5000000,RESOLUTION=1920x1080
1080p.m3u8
```

Each rendition manifest lists 2–10 second chunks. The player watches its own bandwidth and
switches variants between chunks — that's **ABR**.
